<a href="https://colab.research.google.com/github/Arav-18/Walmart-Sales-Dashboard/blob/main/YOLOv11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())

True


# YOLOv11 Baseline — Improved (Target: mAP50 ≥ 0.80)
**Dataset:** NEU-DET Steel Surface Defect (6 classes)
**Model:** YOLOv11n (pretrained on COCO)
**Changes from original:**
- imgsz: 224 → **640** (captures fine defect textures better)
- epochs: 50 → **100** (more training time)
- Added heavy augmentation: mosaic, mixup, copy_paste, degrees
- Added cosine LR scheduler
- Added patience=30 for early stopping
- Added Google Drive auto-save
- Added complete local download with all outputs

## CELL 1 — Setup: Install & Upload Dataset

In [ ]:

import zipfile

zip_path = "NEU-DET.zip"
extract_path = "dataset"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


## CELL 2 — Dataset Preparation (VOC XML → YOLO TXT)

In [ ]:

import os
print(os.listdir("dataset"))
import os
print(os.listdir("dataset/NEU-DET"))

['NEU-DET']
['IMAGES', 'ANNOTATIONS']


In [ ]:
get_ipython().system('pip install xmltodict tqdm')

In [ ]:
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm
import shutil

# Paths
image_dir = "dataset/NEU-DET/IMAGES"
annotation_dir = "dataset/NEU-DET/ANNOTATIONS"

# Output folders
os.makedirs("dataset/yolo/images/train", exist_ok=True)
os.makedirs("dataset/yolo/labels/train", exist_ok=True)

# Class names
classes = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

def convert_bbox(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

for xml_file in tqdm(os.listdir(annotation_dir)):
    tree = ET.parse(os.path.join(annotation_dir, xml_file))
    root = tree.getroot()

    # FIXED filename
    filename = xml_file.replace(".xml", ".jpg")

    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    label_file = os.path.splitext(filename)[0] + ".txt"

    with open(f"dataset/yolo/labels/train/{label_file}", "w") as f:
        for obj in root.iter('object'):
            cls = obj.find('name').text
            if cls not in classes:
                continue
            cls_id = classes.index(cls)

            xmlbox = obj.find('bndbox')
            b = (
                float(xmlbox.find('xmin').text),
                float(xmlbox.find('xmax').text),
                float(xmlbox.find('ymin').text),
                float(xmlbox.find('ymax').text)
            )

            bb = convert_bbox((w, h), b)
            f.write(f"{cls_id} {' '.join(map(str, bb))}\n")


    shutil.copy(
        os.path.join(image_dir, filename),
        os.path.join("dataset/yolo/images/train", filename)
    )

import os

print("Images:", len(os.listdir("dataset/yolo/images/train")))
print("Labels:", len(os.listdir("dataset/yolo/labels/train")))

100%|██████████| 1800/1800 [00:00<00:00, 3618.59it/s]

Images: 1800
Labels: 1800


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.4 MB/s eta 0:00:00


In [ ]:
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm
import shutil
import random

# --- Paths (re-using definitions from previous cells) ---
image_dir = "dataset/NEU-DET/IMAGES"
annotation_dir = "dataset/NEU-DET/ANNOTATIONS"

# --- Class names (re-using definition from previous cells) ---
classes = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

# --- Bounding box conversion function (re-using definition from previous cells) ---
def convert_bbox(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

# --- Preparation: Get all XML files and shuffle ---
all_xml_files = os.listdir(annotation_dir)
random.shuffle(all_xml_files)

# --- Split ratio (e.g., 80% train, 20% val) ---
split_ratio = 0.8
split_index = int(len(all_xml_files) * split_ratio)

train_xml_files = all_xml_files[:split_index]
val_xml_files = all_xml_files[split_index:]

# --- Define the process function ---
def process_dataset_split(xml_file_list, split_type):
    print(f"\nProcessing {split_type} set ({len(xml_file_list)} images)...")

    # Create output folders for the current split type
    os.makedirs(f"dataset/yolo/images/{split_type}", exist_ok=True)
    os.makedirs(f"dataset/yolo/labels/{split_type}", exist_ok=True)

    for xml_file in tqdm(xml_file_list, desc=f"Converting {split_type} set"):
        tree = ET.parse(os.path.join(annotation_dir, xml_file))
        root = tree.getroot()

        # FIXED filename
        filename = xml_file.replace(".xml", ".jpg")

        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        label_file = os.path.splitext(filename)[0] + ".txt"

        with open(f"dataset/yolo/labels/{split_type}/{label_file}", "w") as f:
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in classes:
                    continue
                cls_id = classes.index(cls)

                xmlbox = obj.find('bndbox')
                b = (
                    float(xmlbox.find('xmin').text),
                    float(xmlbox.find('xmax').text),
                    float(xmlbox.find('ymin').text),
                    float(xmlbox.find('ymax').text)
                )

                bb = convert_bbox((w, h), b)
                f.write(f"{cls_id} {' '.join(map(str, bb))}\n")

        # Copy image file
        shutil.copy(
            os.path.join(image_dir, filename),
            os.path.join(f"dataset/yolo/images/{split_type}", filename)
        )


# --- Clean up previously created 'yolo' dataset folder to ensure fresh split ---
if os.path.exists("dataset/yolo"):
    print("Removing existing 'dataset/yolo' folder to re-create split...")
    shutil.rmtree("dataset/yolo")

# --- Process the train and validation splits ---
process_dataset_split(train_xml_files, "train")
process_dataset_split(val_xml_files, "val")

print("\nDataset split and converted successfully!")
print("Images in train set:", len(os.listdir("dataset/yolo/images/train")))
print("Labels in train set:", len(os.listdir("dataset/yolo/labels/train")))
print("Images in val set:", len(os.listdir("dataset/yolo/images/val")))
print("Labels in val set:", len(os.listdir("dataset/yolo/labels/val")))

Removing existing 'dataset/yolo' folder to re-create split...

Processing train set (1440 images)...


Converting train set: 100%|██████████| 1440/1440 [00:00<00:00, 2529.54it/s]



Processing val set (360 images)...


Converting val set: 100%|██████████| 360/360 [00:00<00:00, 2632.38it/s]


Dataset split and converted successfully!
Images in train set: 1440
Labels in train set: 1440
Images in val set: 360
Labels in val set: 360


## CELL 3 — Write data.yaml

In [ ]:
%%writefile /content/dataset/yolo/data.yaml
train: /content/dataset/yolo/images/train
val: /content/dataset/yolo/images/val
nc: 6
names: ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

Writing /content/dataset/yolo/data.yaml


## CELL 4 — Train YOLOv11 Baseline (IMPROVED SETTINGS)

**Key changes from original to push mAP ≥ 0.80:**
| Parameter | Original | Improved | Reason |
|---|---|---|---|
| imgsz | 224 | 640 | Higher resolution = better feature extraction for small defects |
| epochs | 50 | 150 | More training for convergence |
| mosaic | default | 1.0 | Mixes 4 images — better generalization |
| mixup | 0 | 0.15 | Blends images — reduces overfitting |
| copy_paste | 0 | 0.3 | Copies defect regions across images |
| degrees | 0 | 5 | Small rotation augmentation |
| cos_lr | False | True | Cosine LR decay — better final convergence |
| patience | 100 | 30 | Early stop if no improvement after 30 epochs |

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/dataset/yolo/data.yaml",

    # --- Core Settings ---
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,   # use GPU
    name="yolov11_baseline_improved",

    # --- Augmentation (NEW) ---
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    degrees=5,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,

    # --- Optimizer & LR ---
    cos_lr=True,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,

    # --- Training Control ---
    patience=30,
    plots=True,
    save=True,
    exist_ok=False,
    seed=42,
)

print("\nTraining complete!")
print(f"Best mAP50    : {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"Best mAP50-95 : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/yolo/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11_baseline_improved2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

## CELL 5 — Validation with Per-Class Results

## CELL 6 — FPS / Inference Speed Test

In [ ]:
import torch
import time
from ultralytics import YOLO

# Corrected path to the best model weights
model_speed = YOLO("runs/detect/yolov11_baseline_improved2/weights/best.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_speed.model.to(device)
model_speed.model.eval()

# Use same imgsz as training
dummy = torch.randn(1, 3, 640, 640).to(device)

# Warm-up runs
for _ in range(10):
    _ = model_speed.model(dummy)

# Measure FPS
num_images = 200
start = time.time()
with torch.no_grad():
    for _ in range(num_images):
        _ = model_speed.model(dummy)
end = time.time()

fps = num_images / (end - start)
print(f" YOLOv11 Baseline FPS at imgsz=640: {fps:.2f}")
print(f" Inference time per image: {1000/fps:.2f} ms")

 YOLOv11 Baseline FPS at imgsz=640: 73.42
 Inference time per image: 13.62 ms


## CELL 7 — Save All Results to Google Drive (IMPORTANT — Run This First!)

In [ ]:
from google.colab import drive
import shutil, os

# Mount Google Drive
drive.mount('/content/drive')

# Create project folder in Drive
drive_save_path = "/content/drive/MyDrive/SteelProject/YOLOv11_Baseline_Improved"
os.makedirs(drive_save_path, exist_ok=True)

# Copy training results folder
shutil.copytree(
    "runs/detect/yolov11_baseline_improved",
    f"{drive_save_path}/runs_detect_yolov11_baseline_improved",
    dirs_exist_ok=True
)

# Copy val results folder
if os.path.exists("runs/detect/yolov11_baseline_val"):
    shutil.copytree(
        "runs/detect/yolov11_baseline_val",
        f"{drive_save_path}/runs_detect_yolov11_baseline_val",
        dirs_exist_ok=True
    )

print(f" All results saved to Google Drive at:")
print(f"   {drive_save_path}")
print("\nFiles saved:")
for f in os.listdir(f"{drive_save_path}/runs_detect_yolov11_baseline_improved"):
    print(f"  {f}")

Mounted at /content/drive
 All results saved to Google Drive at:
   /content/drive/MyDrive/SteelProject/YOLOv11_Baseline_Improved

Files saved:
  train_batch0.jpg
  weights
  train_batch1.jpg
  labels.jpg
  args.yaml
  train_batch2.jpg


## CELL 8 — Download Everything to Local PC (Complete ZIP)

In [ ]:
import os
import zipfile
from google.colab import files

zip_path = "/content/YOLOv11_Baseline_ALL_RESULTS.zip"

# Folders to include
folders_to_zip = [
    "runs/detect/yolov11_baseline_improved",   # training results
    "runs/detect/yolov11_baseline_val",        # validation results
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in folders_to_zip:
        if not os.path.exists(folder):
            print(f"  Skipping (not found): {folder}")
            continue
        for root, dirs, file_list in os.walk(folder):
            for fname in file_list:
                full_path = os.path.join(root, fname)
                arcname = os.path.relpath(full_path, "/content")
                zipf.write(full_path, arcname)
                print(f"  Added: {arcname}")

print(f"\n Zip created: {zip_path}")
size_mb = os.path.getsize(zip_path) / (1024*1024)
print(f"   Size: {size_mb:.1f} MB")

# Download to local PC
files.download(zip_path)
print("\n Download started — check your browser Downloads folder!")

  Added: runs/detect/yolov11_baseline_improved/train_batch0.jpg
  Added: runs/detect/yolov11_baseline_improved/train_batch1.jpg
  Added: runs/detect/yolov11_baseline_improved/labels.jpg
  Added: runs/detect/yolov11_baseline_improved/args.yaml
  Added: runs/detect/yolov11_baseline_improved/train_batch2.jpg
  Added: runs/detect/yolov11_baseline_improved/weights/yolo11n.pt
  Added: runs/detect/yolov11_baseline_improved/weights/yolo26n.pt

 Zip created: /content/YOLOv11_Baseline_ALL_RESULTS.zip
   Size: 10.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Download started — check your browser Downloads folder!


## CELL 9 — Download Only the Model Weights (Fast Option)

In [ ]:
from google.colab import files
import os

weight_path = "runs/detect/yolov11_baseline_improved/weights/best.pt"

if os.path.exists(weight_path):
    files.download(weight_path)
    print(" best.pt downloaded!")
else:
    print(" best.pt not found — check if training completed.")

## CELL 10 — Print Final Summary Report

In [ ]:
import os
import pandas as pd

results_csv = "runs/detect/yolov11_baseline_improved/results.csv"

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # remove whitespace from column names

    best_epoch = df["metrics/mAP50(B)"].idxmax()
    best_row = df.loc[best_epoch]

    print("="*55)
    print("       YOLOv11 BASELINE — FINAL SUMMARY REPORT")
    print("="*55)
    print(f"  Total Epochs Trained : {len(df)}")
    print(f"  Best Epoch           : {int(best_row['epoch'])}")
    print(f"  Best mAP50           : {best_row['metrics/mAP50(B)']:.4f}")
    print(f"  Best mAP50-95        : {best_row['metrics/mAP50-95(B)']:.4f}")
    print(f"  Best Precision       : {best_row['metrics/precision(B)']:.4f}")
    print(f"  Best Recall          : {best_row['metrics/recall(B)']:.4f}")
    print(f"  Final box_loss       : {best_row['train/box_loss']:.4f}")
    print(f"  Final cls_loss       : {best_row['train/cls_loss']:.4f}")
    print("="*55)
    print("  Saved Files:")
    runs_dir = "runs/detect/yolov11_baseline_improved"
    for f in sorted(os.listdir(runs_dir)):
        size = os.path.getsize(os.path.join(runs_dir, f)) if os.path.isfile(os.path.join(runs_dir, f)) else 0
        if size > 0:
            print(f"    {f:<40} {size/1024:.1f} KB")
    print("="*55)
else:
    print("results.csv not found. Make sure training completed.")

results.csv not found. Make sure training completed.
